# Tarea 2 - Selección, limpieza y alistamiento de datos

## Pregunta de negocio 3

**Pregunta:**

¿Cómo se comporta la ejecución financiera de los contratos de INVIAS y cuáles presentan mayores niveles de recursos pendientes de ejecución, recursos pendientes de pago o extensiones en el plazo contractual que puedan requerir seguimiento?

## Objetivo

Preparar y alistar el conjunto de datos correspondiente a los contratos de INVIAS para realizar posteriormente el análisis descriptivo orientado a responder la pregunta de negocio 3.

## Etapas

1. Auditoría inicial de los datos.
2. Selección de variables relevantes.
3. Identificación y tratamiento de datos faltantes.
4. Identificación y tratamiento de duplicados e inconsistencias.
5. Transformación de variables.
6. Creación de indicadores analíticos.
7. Generación del conjunto de datos final para el análisis.

## 1. Auditoría inicial de los datos

En esta etapa se verifica la estructura general del conjunto de datos
correspondiente a los contratos de INVIAS, identificando el número de
registros y variables disponibles antes de realizar cualquier
transformación o limpieza.

In [2]:
# Importamos librerías necesarias
import pandas as pd

In [3]:
# Importamos la librería os para obtener el directorio de trabajo actual
import os
print(os.getcwd())

/Users/caldeeh/Library/CloudStorage/OneDrive-Personal/1. Maestría Uniandes/3. Materias por semestre/5. Semestre 5/1. Analítica computacional/Proyectos/Proyecto 1/Tarea 2/Pregunta 3


In [4]:
# Cargamos el archivo CSV en un DataFrame de pandas
# El archivo CSV se encuentra en la ruta relativa "../../invias.csv"
df = pd.read_csv("../../invias.csv")

/var/folders/bd/ygh4x6gj0jd4vfxdxdjck_s00000gn/T/ipykernel_2059/2828549143.py:3: DtypeWarning: Columns (0: direccion_de_ejecucion_del_contrato) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../invias.csv")


### 1.1 Dimensiones del conjunto de datos

Se verifica inicialmente el número de registros y variables disponibles
en el conjunto de datos de INVIAS. Esta revisión permite conocer la
dimensión del dataset antes de iniciar los procesos de limpieza y
alistamiento.

In [5]:
# Verificamos el número de registros y variables disponibles
print("Número de filas:", df.shape[0])
print("Número de columnas:", df.shape[1])

Número de filas: 25605
Número de columnas: 89


### 1.2 Exploración de las variables

Se revisan los nombres y tipos de datos de las variables disponibles con
el propósito de identificar aquellas relacionadas con la ejecución
financiera, los plazos contractuales y las características de los
contratos que serán utilizadas posteriormente en el análisis.

In [6]:
# Revisamos el nombre y tipo de dato de cada variable
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25605 entries, 0 to 25604
Data columns (total 89 columns):
 #   Column                                                            Non-Null Count  Dtype  
---  ------                                                            --------------  -----  
 0   id                                                                25605 non-null  str    
 1   version                                                           25605 non-null  str    
 2   created_at                                                        25605 non-null  str    
 3   updated_at                                                        25605 non-null  str    
 4   nombre_entidad                                                    25605 non-null  str    
 5   nit_entidad                                                       25605 non-null  int64  
 6   departamento                                                      25605 non-null  str    
 7   ciudad                                    

### 1.3 Identificación de variables relevantes para la pregunta de negocio 3

A partir de la pregunta de negocio definida, se identifican inicialmente
las variables relacionadas con la ejecución financiera de los contratos,
los plazos contractuales, el estado del contrato, el proveedor, la
modalidad de contratación y la ubicación.

En esta etapa las variables se identifican como candidatas para el
análisis. La selección definitiva se realizará después de revisar su
calidad, completitud y consistencia.

In [7]:
# Definimos las variables inicialmente relevantes para la Pregunta 3

variables_p3 = [
    "id_contrato",
    "referencia_del_contrato",
    "proveedor_adjudicado",
    "tipo_de_contrato",
    "modalidad_de_contratacion",
    "estado_contrato",
    "departamento",
    "ciudad",
    "fecha_de_firma",
    "fecha_de_inicio_del_contrato",
    "fecha_de_fin_del_contrato",
    "duracion_del_contrato",
    "dias_adicionados",
    "valor_del_contrato",
    "valor_facturado",
    "valor_pagado",
    "valor_pendiente_de_pago",
    "valor_pendiente_de_ejecucion",
    "valor_amortizado",
    "saldo_cdp",
    "saldo_vigencia",
    "origen_de_los_recursos",
    "destino_gasto",
    "es_pyme",
    "es_grupo"
]

print("Número de variables candidatas:", len(variables_p3))

Número de variables candidatas: 25


### 1.4 Análisis de datos faltantes

Se analiza la completitud de las variables inicialmente seleccionadas
para la Pregunta de Negocio 3. Para cada variable se calcula el número
de registros con información, el número de valores faltantes y el
porcentaje de faltantes.

En esta etapa no se realiza imputación ni eliminación de registros.
El objetivo es diagnosticar la calidad de los datos antes de definir
una estrategia de tratamiento.

In [8]:
# Calculamos la cantidad y porcentaje de valores faltantes
faltantes_p3 = pd.DataFrame({
    "no_nulos": df[variables_p3].notna().sum(),
    "faltantes": df[variables_p3].isna().sum(),
})

faltantes_p3["porcentaje_faltantes"] = (
    faltantes_p3["faltantes"] / len(df) * 100
)

faltantes_p3 = faltantes_p3.sort_values(
    "porcentaje_faltantes",
    ascending=False
)

faltantes_p3

,no_nulos,faltantes,porcentaje_faltantes
fecha_de_firma,17477,8128,31.743800
fecha_de_inicio_del_contrato,17795,7810,30.501855
fecha_de_fin_del_contrato,20074,5531,21.601250
duracion_del_contrato,20699,4906,19.160320
dias_adicionados,20718,4887,19.086116
valor_pagado,20718,4887,19.086116
es_pyme,20718,4887,19.086116
destino_gasto,20718,4887,19.086116
origen_de_los_recursos,20718,4887,19.086116
saldo_vigencia,20718,4887,19.086116


### 1.5 Análisis del patrón de datos faltantes

La revisión inicial evidencia un patrón común de 4.887 valores faltantes
(19,09 % de los registros) en múltiples variables relacionadas con las
características y la ejecución financiera de los contratos.

Debido a que el número de faltantes coincide en diferentes variables,
se verifica si estos corresponden a los mismos registros antes de
definir cualquier estrategia de eliminación o imputación.

In [9]:
# Identificamos los registros sin información en valor_del_contrato
faltantes_valor = df[df["valor_del_contrato"].isna()]

print("Registros sin valor del contrato:", len(faltantes_valor))

Registros sin valor del contrato: 4887


In [10]:
# Revisamos la completitud de variables clave dentro de los registros que no tienen información en valor_del_contrato

variables_revision = [
    "proveedor_adjudicado",
    "tipo_de_contrato",
    "modalidad_de_contratacion",
    "dias_adicionados",
    "valor_facturado",
    "valor_pagado",
    "valor_pendiente_de_pago",
    "valor_pendiente_de_ejecucion"
]

faltantes_valor[variables_revision].notna().sum()

proveedor_adjudicado            0
tipo_de_contrato                0
modalidad_de_contratacion       0
dias_adicionados                0
valor_facturado                 0
valor_pagado                    0
valor_pendiente_de_pago         0
valor_pendiente_de_ejecucion    0
dtype: int64

In [11]:
# Analizamos el estado de los registros sin valor del contrato
faltantes_valor["estado_contrato"].value_counts(dropna=False)

estado_contrato
Cerrado              1641
Modificado           1302
En ejecución          792
terminado             470
Borrador              307
Suspendido            139
Aprobado               99
Cancelado              51
enviado Proveedor      51
cedido                 31
En aprobación           4
Name: count, dtype: int64

### 1.6 Caracterización de los registros con información financiera faltante

Se caracterizan los registros que presentan ausencia de información en
`valor_del_contrato` y en las principales variables financieras. El
objetivo es determinar si corresponden a registros contractuales
válidos con información incompleta o si presentan alguna característica
particular que permita explicar el patrón de datos faltantes.

In [12]:
# Revisamos una muestra de los registros sin información financiera

faltantes_valor[
    [
        "id",
        "id_contrato",
        "referencia_del_contrato",
        "estado_contrato",
        "departamento",
        "ciudad"
    ]
].head(10)

,id,id_contrato,referencia_del_contrato,estado_contrato,departamento,ciudad
5,row-djnx.jude~xszd,CO1.PCCNTR.9279238,1150-2026,Aprobado,Distrito Capital de Bogotá,Bogotá
6,row-zyzi~adhy_t5fd,CO1.PCCNTR.3241665,303-2022,Cerrado,Distrito Capital de Bogotá,Bogotá
9,row-htxd-wzm9.sdz9,CO1.PCCNTR.9272734,1141-2026,En ejecución,Distrito Capital de Bogotá,Bogotá
14,row-gamy-94m8~vim6,CO1.PCCNTR.3012807,1773-2021,Modificado,Distrito Capital de Bogotá,Bogotá
27,row-fuab_knt2~rezy,CO1.PCCNTR.8747985,2213-2025.,En ejecución,Distrito Capital de Bogotá,Bogotá
28,row-7eri~64z4-hyjr,CO1.PCCNTR.5273883,2920-2023,terminado,Distrito Capital de Bogotá,Bogotá
31,row-w7sv-wa3a_yfg9,CO1.PCCNTR.2192385,509-2021,Cerrado,Distrito Capital de Bogotá,Bogotá
37,row-xhzp.bmtv.2zqa,CO1.PCCNTR.9272917,1144-2026,Borrador,Distrito Capital de Bogotá,Bogotá
38,row-g6gf~vbs4.8tv6,CO1.PCCNTR.6561364,2639-2024,En ejecución,Distrito Capital de Bogotá,Bogotá
39,row-tj7d-77j4~5tg7,CO1.PCCNTR.9261166,1058-2026,Modificado,Distrito Capital de Bogotá,Bogotá


### 1.7 Verificación de unicidad de los contratos

Se verifica la unicidad de los identificadores contractuales con el fin
de determinar si cada registro corresponde a un contrato diferente o si
existen múltiples registros asociados a un mismo contrato.

Esta revisión es necesaria antes de definir el tratamiento de los
registros con información financiera faltante.

In [13]:
# Verificamos la cantidad de identificadores de contrato únicos

print("Registros totales:", len(df))
print("ID de contrato únicos:", df["id_contrato"].nunique())
print("Referencias de contrato únicas:", df["referencia_del_contrato"].nunique())

Registros totales: 25605
ID de contrato únicos: 25605
Referencias de contrato únicas: 24476


### 1.8 Análisis de referencias de contrato repetidas

Aunque el identificador `id_contrato` es único para todos los registros,
se identificaron referencias de contrato repetidas. Se analiza la
frecuencia de estas referencias para determinar si corresponden a
múltiples registros contractuales asociados a una misma referencia.

In [14]:
# Identificamos las referencias de contrato que aparecen más de una vez

frecuencia_referencias = (
    df["referencia_del_contrato"]
    .value_counts()
)

frecuencia_referencias[frecuencia_referencias > 1].head(20)

referencia_del_contrato
.               12
CANCELADO        7
-                6
3655-2023        6
0                5
0079-2022        4
4616-2023        4
4239-2023        4
0385-2025        4
4528-2023        4
1941 DE 2024     4
4134-2023        4
0128-2022        4
468-2022         4
0316-2025        4
XXXX             4
2408-2025        4
4065 DE 2024     4
0309-2025        4
1913-2025        4
Name: count, dtype: int64

### 1.9 Caracterización de referencias contractuales repetidas

Se selecciona una referencia contractual repetida como caso de
inspección para identificar las diferencias entre los registros
asociados a una misma referencia y determinar si la repetición
corresponde a registros contractuales distintos o a una característica
propia de la fuente de datos.

In [15]:
# Revisamos los registros asociados a una referencia contractual repetida

df[df["referencia_del_contrato"] == "3655-2023"][
    [
        "id",
        "id_contrato",
        "referencia_del_contrato",
        "estado_contrato",
        "fecha_de_firma",
        "valor_del_contrato",
        "proveedor_adjudicado"
    ]
]

,id,id_contrato,referencia_del_contrato,estado_contrato,fecha_de_firma,valor_del_contrato,proveedor_adjudicado
5646,row-cc7e~riru_vapt,CO1.PCCNTR.5436160,3655-2023,Cancelado,NaN,0.0,Sin Descripcion
9426,row-c6e5.qkw8_qvuz,CO1.PCCNTR.5444652,3655-2023,enviado Proveedor,NaN,NaN,NaN
12109,row-3azc_y3jx~vdqz,CO1.PCCNTR.5427513,3655-2023,enviado Proveedor,NaN,200164946.0,JUNTA DE ACCION COMUNAL VEREDA GUARICA
20082,row-6y9f-sphj~ag9f,CO1.PCCNTR.5445466,3655-2023,Modificado,NaN,NaN,NaN
20926,row-iquw~7pv6-9hyr,CO1.PCCNTR.5436886,3655-2023,enviado Proveedor,NaN,200164946.0,JUNTA DE ACCION COMUNAL VEREDA GUARICA
24711,row-ca8h~skpk~4hbb,CO1.PCCNTR.5441490,3655-2023,enviado Proveedor,NaN,200164946.0,JUNTA DE ACCION COMUNAL VEREDA GUARICA


### 1.10 Comparación de los registros con y sin información financiera

Se compara la estructura de los identificadores contractuales entre los
registros que presentan información financiera y aquellos que no
presentan valor en `valor_del_contrato`, con el propósito de identificar
si ambos grupos pertenecen al mismo tipo de registro.

In [16]:
# Comparamos algunos identificadores de los dos grupos de registros

print("Ejemplo de contratos SIN valor:")
print(faltantes_valor["id_contrato"].head(5).to_list())

print("\nEjemplo de contratos CON valor:")
print(df[df["valor_del_contrato"].notna()]["id_contrato"].head(5).to_list())

Ejemplo de contratos SIN valor:
['CO1.PCCNTR.9279238', 'CO1.PCCNTR.3241665', 'CO1.PCCNTR.9272734', 'CO1.PCCNTR.3012807', 'CO1.PCCNTR.8747985']

Ejemplo de contratos CON valor:
['CO1.PCCNTR.770322', 'CO1.PCCNTR.5481469', 'CO1.PCCNTR.6706615', 'CO1.PCCNTR.4412417', 'CO1.PCCNTR.3516681']


### 1.11 Distribución temporal de los registros con información financiera faltante

Se analiza la distribución de los registros sin información en
`valor_del_contrato` según el año identificado en la referencia
contractual. El objetivo es determinar si los valores faltantes se
concentran en determinados periodos.

In [17]:
# Extraemos el año de la referencia contractual cuando esta termina en un año de cuatro dígitos

faltantes_valor["año_referencia"] = (
    faltantes_valor["referencia_del_contrato"]
    .str.extract(r"(\d{4})$")[0]
)

faltantes_valor["año_referencia"].value_counts().sort_index()

año_referencia
0147    1
0188    1
0224    1
0515    1
0623    1
       ..
9399    1
9441    1
9524    1
9579    1
9742    1
Name: count, Length: 75, dtype: int64

### 1.12 Revisión del formato de las variables de fecha

Se inspeccionan algunos valores de las variables relacionadas con las
fechas contractuales para identificar su formato actual y determinar el
tratamiento requerido para su conversión a variables de tipo fecha.

In [18]:
# Revisamos ejemplos de las principales variables de fecha

df[
    [
        "fecha_de_firma",
        "fecha_de_inicio_del_contrato",
        "fecha_de_fin_del_contrato"
    ]
].head(10)

,fecha_de_firma,fecha_de_inicio_del_contrato,fecha_de_fin_del_contrato
0,2019-01-29T00:00:00.000,2019-01-18T00:00:00.000,2019-07-17T00:00:00.000
1,2023-10-23T00:00:00.000,2023-10-31T00:00:00.000,2023-12-31T00:00:00.000
2,2024-08-30T00:00:00.000,2024-09-06T00:00:00.000,2024-12-31T00:00:00.000
3,2023-01-16T00:00:00.000,2023-01-19T00:00:00.000,2023-04-19T00:00:00.000
4,NaN,NaN,2022-09-27T00:00:00.000
5,NaN,NaN,NaN
6,NaN,NaN,NaN
7,2025-09-12T00:00:00.000,2025-09-26T00:00:00.000,2029-01-01T00:00:00.000
8,2023-01-19T00:00:00.000,2023-02-01T00:00:00.000,2023-04-30T00:00:00.000
9,NaN,NaN,NaN


### 1.13 Conversión de variables de fecha

Las variables `fecha_de_firma`, `fecha_de_inicio_del_contrato` y
`fecha_de_fin_del_contrato` se encuentran almacenadas como texto y
presentan el formato ISO `AAAA-MM-DDTHH:MM:SS.sss`.

Para facilitar el análisis temporal, se realiza una conversión a tipo
`datetime`. En esta etapa se conservan las variables originales y se
crean nuevas variables para validar el resultado de la transformación.

In [19]:
# Convertimos las variables de fecha a formato datetime y las almacenamos temporalmente en nuevas columnas

df["fecha_firma_dt"] = pd.to_datetime(
    df["fecha_de_firma"],
    errors="coerce"
)

df["fecha_inicio_dt"] = pd.to_datetime(
    df["fecha_de_inicio_del_contrato"],
    errors="coerce"
)

df["fecha_fin_dt"] = pd.to_datetime(
    df["fecha_de_fin_del_contrato"],
    errors="coerce"
)

### 1.14 Validación de la conversión de fechas

Se verifica el resultado de la conversión de las variables de fecha,
comparando la cantidad de valores disponibles antes y después de la
transformación. Esto permite identificar si existían valores que no
pudieron convertirse correctamente.

In [20]:
# Comparamos los valores disponibles antes y después de la conversión

print("Fecha de firma:")
print("  Original:", df["fecha_de_firma"].notna().sum())
print("  Convertida:", df["fecha_firma_dt"].notna().sum())

print("\nFecha de inicio:")
print("  Original:", df["fecha_de_inicio_del_contrato"].notna().sum())
print("  Convertida:", df["fecha_inicio_dt"].notna().sum())

print("\nFecha de fin:")
print("  Original:", df["fecha_de_fin_del_contrato"].notna().sum())
print("  Convertida:", df["fecha_fin_dt"].notna().sum())

Fecha de firma:
  Original: 17477
  Convertida: 17477

Fecha de inicio:
  Original: 17795
  Convertida: 17795

Fecha de fin:
  Original: 20074
  Convertida: 20074


### 1.15 Caracterización temporal de los registros sin valor del contrato

Se analiza la disponibilidad de fechas contractuales en los registros que
no presentan información en `valor_del_contrato`. El objetivo es
determinar si estos registros corresponden a contratos para los cuales
existe información temporal a pesar de la ausencia de información
financiera.

In [21]:
# Actualizamos el subconjunto de registros sin valor del contrato para incluir las nuevas variables de fecha convertidas

faltantes_valor = df[df["valor_del_contrato"].isna()].copy()

print("Registros sin valor del contrato:", len(faltantes_valor))

Registros sin valor del contrato: 4887


In [22]:
# Revisamos la disponibilidad de fechas en los registros que no tienen valor del contrato

print("Registros sin valor del contrato:", len(faltantes_valor))

print("\nFecha de firma disponible:",
      faltantes_valor["fecha_firma_dt"].notna().sum())

print("Fecha de inicio disponible:",
      faltantes_valor["fecha_inicio_dt"].notna().sum())

print("Fecha de fin disponible:",
      faltantes_valor["fecha_fin_dt"].notna().sum())

Registros sin valor del contrato: 4887

Fecha de firma disponible: 0
Fecha de inicio disponible: 0
Fecha de fin disponible: 0


### Hallazgo sobre los registros con información faltante

Se identificaron 4.887 registros, equivalentes al 19,09 % del conjunto
de datos, que no presentan información en `valor_del_contrato` ni en
las principales variables financieras analizadas. Adicionalmente,
ninguno de estos registros presenta información en `fecha_de_firma`,
`fecha_de_inicio_del_contrato` o `fecha_de_fin_del_contrato`.

Estos registros no serán eliminados automáticamente. Primero se
investigarán sus características adicionales para determinar su origen
y definir una estrategia de tratamiento consistente con el objetivo
del análisis.

### 1.16 Revisión de información de actualización de los registros

Se revisan las variables asociadas a la creación, actualización y
versión de los registros que no presentan información contractual y
financiera, con el propósito de identificar posibles diferencias en
su estructura o estado de actualización.

In [23]:
# Revisamos versión y fechas de actualización de los registros sin información contractual y financiera

faltantes_valor[
    [
        "id_contrato",
        "version",
        "created_at",
        "updated_at",
        "estado_contrato"
    ]
].head(10)

,id_contrato,version,created_at,updated_at,estado_contrato
5,CO1.PCCNTR.9279238,rv-dpch-s8t7~f32f,2026-08-23T08:01:59.248Z,2026-08-23T08:01:59.248Z,Aprobado
6,CO1.PCCNTR.3241665,rv-idyw-5y27~bpga,2026-08-23T08:01:59.248Z,2026-08-23T08:01:59.248Z,Cerrado
9,CO1.PCCNTR.9272734,rv-2s3v_wee5_c3yk,2026-08-23T08:01:59.248Z,2026-08-23T08:01:59.248Z,En ejecución
14,CO1.PCCNTR.3012807,rv-87ce.vnma.natz,2026-08-23T08:01:59.248Z,2026-08-23T08:01:59.248Z,Modificado
27,CO1.PCCNTR.8747985,rv-jyr2.j6td_ahgi,2026-08-23T08:01:59.248Z,2026-08-23T08:01:59.248Z,En ejecución
28,CO1.PCCNTR.5273883,rv-v8re-rdnn.7643,2026-08-23T08:01:59.248Z,2026-08-23T08:01:59.248Z,terminado
31,CO1.PCCNTR.2192385,rv-n8c7.525s-56xn,2026-08-23T08:01:59.248Z,2026-08-23T08:01:59.248Z,Cerrado
37,CO1.PCCNTR.9272917,rv-pxw4~pqjb_i92q,2026-08-23T08:01:59.248Z,2026-08-23T08:01:59.248Z,Borrador
38,CO1.PCCNTR.6561364,rv-as7m.ienp~52xi,2026-08-23T08:01:59.248Z,2026-08-23T08:01:59.248Z,En ejecución
39,CO1.PCCNTR.9261166,rv-td92~2g4e-bcdp,2026-08-23T08:01:59.248Z,2026-08-23T08:01:59.248Z,Modificado


### 1.17 Comparación del estado contractual según disponibilidad de información financiera

Se compara la distribución del `estado_contrato` entre los registros
con y sin información en `valor_del_contrato`. El objetivo es
determinar si la ausencia de información financiera está asociada con
determinados estados contractuales.

In [24]:
# Creamos una tabla de comparación entre registros con y sin valor del contrato

comparacion_estado = pd.crosstab(
    df["estado_contrato"],
    df["valor_del_contrato"].isna(),
    margins=True
)

comparacion_estado

valor_del_contrato,False,True,All
estado_contrato,,,
Aprobado,773,99,872
Borrador,2131,307,2438
Cancelado,578,51,629
Cerrado,7622,1641,9263
En aprobación,283,4,287
En ejecución,1720,792,2512
Modificado,4132,1302,5434
Suspendido,222,139,361
cedido,61,31,92


### 1.18 Revisión temporal de la creación y actualización de registros

Se analiza la distribución de las fechas de creación y actualización de
los registros que no presentan información financiera, con el propósito
de identificar patrones comunes en su incorporación o actualización
dentro de la fuente de datos.

In [25]:
# Revisamos las fechas de creación y actualización de los registros sin información financiera

print("Fechas de creación:")
print(faltantes_valor["created_at"].value_counts().head(10))

print("\nFechas de actualización:")
print(faltantes_valor["updated_at"].value_counts().head(10))

Fechas de creación:
created_at
2026-08-23T08:01:59.248Z    4887
Name: count, dtype: int64

Fechas de actualización:
updated_at
2026-08-23T08:01:59.248Z    4887
Name: count, dtype: int64


In [26]:
# Revisamos la cantidad de versiones únicas en los registros sin información financiera

print("Versiones únicas:", faltantes_valor["version"].nunique())

print("\nVersiones repetidas:")
print(
    faltantes_valor["version"]
    .value_counts()
    .head(10)
)

Versiones únicas: 4887

Versiones repetidas:
version
rv-dpch-s8t7~f32f    1
rv-idyw-5y27~bpga    1
rv-2s3v_wee5_c3yk    1
rv-87ce.vnma.natz    1
rv-jyr2.j6td_ahgi    1
rv-v8re-rdnn.7643    1
rv-n8c7.525s-56xn    1
rv-pxw4~pqjb_i92q    1
rv-as7m.ienp~52xi    1
rv-td92~2g4e-bcdp    1
Name: count, dtype: int64


### 1.19 Selección preliminar de registros con información financiera

A partir de la auditoría de datos faltantes, se genera un subconjunto
preliminar compuesto por los registros que cuentan con información en
`valor_del_contrato`.

Este subconjunto se utilizará para evaluar la completitud de las demás
variables financieras antes de definir el conjunto de datos definitivo
para la Pregunta de Negocio 3.

In [27]:
# Seleccionamos los registros que tienen información en valor_del_contrato

df_financiero = df[df["valor_del_contrato"].notna()].copy()

print("Registros del subconjunto financiero:", len(df_financiero))

Registros del subconjunto financiero: 20718


### 1.20 Completitud de las variables financieras

Se evalúa la disponibilidad de información en las principales variables
financieras dentro del subconjunto preliminar de 20.718 registros.

Esta revisión permite determinar si existen datos faltantes adicionales
una vez excluidos los registros que no contienen información en
`valor_del_contrato`.

In [28]:
# Revisamos la completitud de las principales variables financieras

variables_financieras = [
    "valor_del_contrato",
    "valor_facturado",
    "valor_pagado",
    "valor_pendiente_de_pago",
    "valor_pendiente_de_ejecucion",
    "valor_amortizado",
    "saldo_cdp",
    "saldo_vigencia"
]

completitud_financiera = pd.DataFrame({
    "no_nulos": df_financiero[variables_financieras].notna().sum(),
    "faltantes": df_financiero[variables_financieras].isna().sum()
})

completitud_financiera["porcentaje_faltantes"] = (
    completitud_financiera["faltantes"]
    / len(df_financiero)
    * 100
)

completitud_financiera

,no_nulos,faltantes,porcentaje_faltantes
valor_del_contrato,20718,0,0.0
valor_facturado,20718,0,0.0
valor_pagado,20718,0,0.0
valor_pendiente_de_pago,20718,0,0.0
valor_pendiente_de_ejecucion,20718,0,0.0
valor_amortizado,20718,0,0.0
saldo_cdp,20718,0,0.0
saldo_vigencia,20718,0,0.0


### 1.21 Validación de consistencia de los valores financieros

Se realizan verificaciones de consistencia sobre las variables
financieras con el propósito de identificar posibles valores atípicos
o relaciones que requieran revisión antes de calcular indicadores de
ejecución.

In [29]:
# Identificamos contratos cuyo valor pagado supera el valor contratado

pagado_mayor_contrato = df_financiero[
    df_financiero["valor_pagado"] > df_financiero["valor_del_contrato"]
]

print("Contratos donde valor_pagado > valor_del_contrato:",
      len(pagado_mayor_contrato))

Contratos donde valor_pagado > valor_del_contrato: 5


In [30]:
# Identificamos los contratos que presentan ambas condiciones

casos_ambas_condiciones = df_financiero[
    (df_financiero["valor_pagado"] > df_financiero["valor_del_contrato"]) &
    (df_financiero["valor_facturado"] > df_financiero["valor_del_contrato"])
]

print(
    "Contratos donde valor_pagado y valor_facturado "
    "superan el valor del contrato:",
    len(casos_ambas_condiciones)
)

Contratos donde valor_pagado y valor_facturado superan el valor del contrato: 5


### 1.22 Revisión de registros con posibles inconsistencias financieras

Se inspeccionan individualmente los registros en los cuales tanto el
`valor_facturado` como el `valor_pagado` superan el `valor_del_contrato`.
El objetivo es caracterizar estos casos antes de determinar si
corresponden a inconsistencias de información o a situaciones
contractuales particulares.

In [31]:
# Mostramos los 5 contratos que presentan ambas condiciones

casos_ambas_condiciones[
    [
        "id_contrato",
        "referencia_del_contrato",
        "estado_contrato",
        "proveedor_adjudicado",
        "valor_del_contrato",
        "valor_facturado",
        "valor_pagado",
        "valor_pendiente_de_pago",
        "valor_pendiente_de_ejecucion",
        "dias_adicionados"
    ]
]

,id_contrato,referencia_del_contrato,estado_contrato,proveedor_adjudicado,valor_del_contrato,valor_facturado,valor_pagado,valor_pendiente_de_pago,valor_pendiente_de_ejecucion,dias_adicionados
923,CO1.PCCNTR.5755707,033-2024,cedido,FERNANDO CAICEDO OCHOA,13566666.66,57596000.0,57596000.0,-44029333.0,-44029333.0,0.0
3896,CO1.PCCNTR.828527,543 de 2019,terminado,Marcos Jose Gomez Calderon,64544330.00,64750330.0,64750330.0,-206000.0,-206000.0,0.0
14383,CO1.PCCNTR.1300022,000286 de 2020,terminado,FRANCISCO ANDRADE VARGAS,86948047.67,93386666.0,93386666.0,-6438619.0,-6438619.0,0.0
21957,CO1.PCCNTR.760010,0234-20019,terminado,Daniel Armando Bula Calderón,41999999.66,42000000.0,42000000.0,0.0,0.0,0.0
24534,CO1.PCCNTR.2174192,391 de 2021,terminado,WILLIAM ALBERTO CORDOBA MESA,57141447.00,57170047.0,57170047.0,-28600.0,-28600.0,0.0


### Resultado de la revisión de registros con posibles inconsistencias financieras

Como parte de la auditoría inicial se identificaron 5 contratos en los cuales
el valor facturado y el valor pagado superan simultáneamente el valor del
contrato registrado en la base de datos.

Estos cinco registros representan el 0,024 % de los 20.718 contratos del
subconjunto financiero.

Los casos fueron inspeccionados individualmente considerando el estado del
contrato, proveedor adjudicado y las principales variables financieras.

En los cinco casos identificados, el valor pagado coincide con el valor
facturado. Adicionalmente, se observan valores negativos en las variables
`valor_pendiente_de_pago` y `valor_pendiente_de_ejecucion`.

Estos registros se clasifican como posibles señales de alerta que requieren revisión
posterior, debido a que pueden corresponder a modificaciones contractuales,
particularidades de ejecución o inconsistencias en el registro de la
información financiera.

### Magnitud del exceso sobre el valor contractual

Para los contratos identificados como posibles inconsistencias financieras,
se calcula la diferencia entre el valor facturado/pagado y el valor del
contrato.

También se calcula el porcentaje que representa dicho exceso respecto al
valor contractual, con el propósito de dimensionar la magnitud de cada caso.

In [32]:
# Calculamos la magnitud del exceso financiero

casos_ambas_condiciones = casos_ambas_condiciones.copy()

casos_ambas_condiciones["exceso_facturado"] = (
    casos_ambas_condiciones["valor_facturado"]
    - casos_ambas_condiciones["valor_del_contrato"]
)

casos_ambas_condiciones["porcentaje_exceso_facturado"] = (
    casos_ambas_condiciones["exceso_facturado"]
    / casos_ambas_condiciones["valor_del_contrato"]
) * 100

casos_ambas_condiciones["exceso_pagado"] = (
    casos_ambas_condiciones["valor_pagado"]
    - casos_ambas_condiciones["valor_del_contrato"]
)

casos_ambas_condiciones["porcentaje_exceso_pagado"] = (
    casos_ambas_condiciones["exceso_pagado"]
    / casos_ambas_condiciones["valor_del_contrato"]
) * 100

casos_ambas_condiciones[
    [
        "id_contrato",
        "referencia_del_contrato",
        "proveedor_adjudicado",
        "valor_del_contrato",
        "valor_facturado",
        "exceso_facturado",
        "porcentaje_exceso_facturado",
        "valor_pagado",
        "exceso_pagado",
        "porcentaje_exceso_pagado"
    ]
]

,id_contrato,referencia_del_contrato,proveedor_adjudicado,valor_del_contrato,valor_facturado,exceso_facturado,porcentaje_exceso_facturado,valor_pagado,exceso_pagado,porcentaje_exceso_pagado
923,CO1.PCCNTR.5755707,033-2024,FERNANDO CAICEDO OCHOA,13566666.66,57596000.0,44029333.34,3.245405e+02,57596000.0,44029333.34,3.245405e+02
3896,CO1.PCCNTR.828527,543 de 2019,Marcos Jose Gomez Calderon,64544330.00,64750330.0,206000.00,3.191605e-01,64750330.0,206000.00,3.191605e-01
14383,CO1.PCCNTR.1300022,000286 de 2020,FRANCISCO ANDRADE VARGAS,86948047.67,93386666.0,6438618.33,7.405133e+00,93386666.0,6438618.33,7.405133e+00
21957,CO1.PCCNTR.760010,0234-20019,Daniel Armando Bula Calderón,41999999.66,42000000.0,0.34,8.095238e-07,42000000.0,0.34,8.095238e-07
24534,CO1.PCCNTR.2174192,391 de 2021,WILLIAM ALBERTO CORDOBA MESA,57141447.00,57170047.0,28600.00,5.005124e-02,57170047.0,28600.00,5.005124e-02


### Hallazgo

La magnitud del exceso presenta una alta heterogeneidad entre los cinco
registros. Un caso presenta un exceso del 324,54 % sobre el valor
contractual, mientras que los demás presentan diferencias inferiores al
7,5 %.

Por lo anterior, los registros no deben interpretarse de manera homogénea.
El caso con referencia 033-2024 constituye la principal señal de alerta
dentro de esta revisión y deberá conservarse para análisis posteriores.

Estos resultados representan señales de calidad o consistencia de la
información financiera y no permiten, por sí solos, establecer la
existencia de corrupción o irregularidad contractual.

In [33]:
# Revisión de consistencia entre las principales variables financieras

print(
    "Pagado mayor que facturado:",
    (df_financiero["valor_pagado"] > df_financiero["valor_facturado"]).sum()
)

print(
    "Amortizado mayor que valor del contrato:",
    (df_financiero["valor_amortizado"] > df_financiero["valor_del_contrato"]).sum()
)

# Variables financieras que serán revisadas por valores negativos
variables_financieras = [
    "valor_del_contrato",
    "valor_facturado",
    "valor_pagado",
    "valor_pendiente_de_pago",
    "valor_pendiente_de_ejecucion",
    "valor_amortizado",
    "saldo_cdp",
    "saldo_vigencia"
]

print("\nValores negativos por variable:")

for variable in variables_financieras:
    negativos = (df_financiero[variable] < 0).sum()
    print(f"{variable}: {negativos}")

Pagado mayor que facturado: 0
Amortizado mayor que valor del contrato: 0

Valores negativos por variable:
valor_del_contrato: 0
valor_facturado: 0
valor_pagado: 0
valor_pendiente_de_pago: 4
valor_pendiente_de_ejecucion: 4
valor_amortizado: 0
saldo_cdp: 0
saldo_vigencia: 0


### 1.24 Revisión de valores pendientes negativos

Se identificaron cuatro registros con valores negativos tanto en
`valor_pendiente_de_pago` como en `valor_pendiente_de_ejecucion`.

Se procede a identificar los contratos asociados con estos valores para
determinar si corresponden a los casos previamente identificados por
exceder el valor facturado y pagado respecto al valor contractual.

In [34]:
# Identificamos los contratos con valores pendientes negativos

casos_pendientes_negativos = df_financiero[
    (df_financiero["valor_pendiente_de_pago"] < 0) |
    (df_financiero["valor_pendiente_de_ejecucion"] < 0)
]

casos_pendientes_negativos[
    [
        "id_contrato",
        "referencia_del_contrato",
        "estado_contrato",
        "proveedor_adjudicado",
        "valor_del_contrato",
        "valor_facturado",
        "valor_pagado",
        "valor_pendiente_de_pago",
        "valor_pendiente_de_ejecucion"
    ]
]

,id_contrato,referencia_del_contrato,estado_contrato,proveedor_adjudicado,valor_del_contrato,valor_facturado,valor_pagado,valor_pendiente_de_pago,valor_pendiente_de_ejecucion
923,CO1.PCCNTR.5755707,033-2024,cedido,FERNANDO CAICEDO OCHOA,13566666.66,57596000.0,57596000.0,-44029333.0,-44029333.0
3896,CO1.PCCNTR.828527,543 de 2019,terminado,Marcos Jose Gomez Calderon,64544330.00,64750330.0,64750330.0,-206000.0,-206000.0
14383,CO1.PCCNTR.1300022,000286 de 2020,terminado,FRANCISCO ANDRADE VARGAS,86948047.67,93386666.0,93386666.0,-6438619.0,-6438619.0
24534,CO1.PCCNTR.2174192,391 de 2021,terminado,WILLIAM ALBERTO CORDOBA MESA,57141447.00,57170047.0,57170047.0,-28600.0,-28600.0


### 1.25 Conclusiones de la auditoría financiera

Sobre los 20.718 registros que conforman el subconjunto financiero se
realizaron pruebas de consistencia sobre las principales variables
económicas del contrato.

Se identificaron 5 contratos en los cuales el valor facturado y el valor
pagado superan simultáneamente el valor del contrato. De estos casos, uno
presenta un exceso particularmente elevado, correspondiente al contrato
con referencia 033-2024, cuyo valor facturado y pagado supera en 324,54 %
el valor contractual registrado.

Adicionalmente, se identificaron 4 registros con valores negativos en
`valor_pendiente_de_pago` y `valor_pendiente_de_ejecucion`. Estos cuatro
registros corresponden a cuatro de los cinco contratos previamente
identificados por exceso financiero.

No se encontraron casos en los que el valor pagado supere el valor
facturado, ni casos en los que el valor amortizado supere el valor del
contrato. Tampoco se identificaron valores negativos en las variables
`valor_del_contrato`, `valor_facturado`, `valor_pagado`, `valor_amortizado`,
`saldo_cdp` o `saldo_vigencia`.

Los hallazgos anteriores se consideran señales de alerta sobre la
consistencia de la información financiera. Estos casos deberán
conservarse para análisis posteriores y contrastarse con otras variables
del proceso contractual.

## Resumen de hallazgos de auditoría inicial

| Aspecto auditado | Resultado |
|---|---:|
| Registros originales | 25.605 |
| Registros subconjunto financiero | 20.718 |
| Variables financieras revisadas | 8 |
| Pagado > facturado | 0 |
| Amortizado > valor contrato | 0 |
| Valor contrato negativo | 0 |
| Valor facturado negativo | 0 |
| Valor pagado negativo | 0 |
| Saldo CDP negativo | 0 |
| Saldo vigencia negativo | 0 |
| Contratos con facturado > contrato | 5 |
| Contratos con pagado > contrato | 5 |
| Contratos con ambas condiciones | 5 |
| Contratos con pendientes negativos | 4 |

### Decisión para la siguiente etapa

Los resultados de la auditoría inicial serán utilizados como insumo para
definir las reglas de limpieza y transformación de los datos. Los cinco
casos con posibles inconsistencias financieras no serán eliminados, ya
que constituyen observaciones relevantes para el análisis posterior de
riesgo contractual.